# Test: skjuvbuckling

Kontrollerar `kb.skjuvbuckling` mot EN 1993-1-5 kapitel 5 och bilaga A.3.

Kör med **Run All**. Kerneln ska vara `.venv` i repot.

- ✔ kontrollen gick igenom
- ✘ känt fel: fel som redan är kända och inte rättade än
- ✘ FEL: nytt fel

In [ ]:
import math
import kthbygg as kb
from kontroll import kolla, kolla_fel, klass, utskrift, sammanfattning, nollställ

nollställ()

## η och plastisk skjuvbärförmåga

In [ ]:
kolla("η S355", lambda: kb.skjuvbuckling.η(355), 1.2)
kolla("η f_y > 460", lambda: kb.skjuvbuckling.η(500), 1.0, känt_fel=True)
kolla("eta f_y > 460", lambda: kb.skjuvbuckling.eta(500), 1.0)
kolla("V_pl,Rd A_v = 3000 mm², S355", lambda: kb.skjuvbuckling.V_pl_Rd(355, 1.0, 3000), 614878)

## Bucklingskoefficient k_τ

Utan längsavstyvning: a/h_w ≥ 1 ger k_τ = 5,34 + 4(h_w/a)², annars 4 + 5,34(h_w/a)².

Med längsavstyvning: k_τsl = 9(h_w/a)² ⁴√((I_sl/(t³h_w))³) ≥ (2,1/t)∛(I_sl/h_w). Med h_w = 1000, a = 2000, I_sl = 10⁶, t = 10 blir k_τsl = 2,25 och k_τ = 8,59.

In [ ]:
kolla("k_τ a/h_w = 2", lambda: kb.skjuvbuckling.k_τ(1000, 2000), 6.34)
kolla("k_τ a/h_w = 0,8", lambda: kb.skjuvbuckling.k_τ(1000, 800), 12.34375)
kolla("k_τ med längsavstyvning", lambda: kb.skjuvbuckling.k_τ(1000, 2000, I_st=1e6, t=10), 8.59, känt_fel=True)

## Slankhet λ̄_w

h_w = 1000, t_w = 8, S355, k_τ = 6,34. Formeln med 37,4 och vägen via τ_cr ska ge samma värde.

In [ ]:
ε = math.sqrt(235 / 355)
kolla("λ_w", lambda: kb.skjuvbuckling.λ_w(1000, 8, ε, 6.34), 1.63145)
τ_cr = kolla("τ_cr", lambda: kb.skjuvbuckling.τ_cr(8, 1000, 6.34), 77.0133)
kolla("λ_w via τ_cr", lambda: kb.skjuvbuckling.λ_w_med_τ_cr(355, τ_cr), 1.63145)
kolla("λ_w förenklad", lambda: kb.skjuvbuckling.λ_w_förenklad(1000, 8, ε), 1000 / (86.4 * ε * 8))

## Reduktionsfaktor χ_w (tabell 5.1)

In [ ]:
kolla("χ_w λ̄_w = 0,5", lambda: kb.skjuvbuckling.χ_w(0.5, 1.2), 1.2)
kolla("χ_w λ̄_w = 1,0", lambda: kb.skjuvbuckling.χ_w(1.0, 1.2), 0.83)
kolla("χ_w λ̄_w = 1,5, styv ändavstyvning", lambda: kb.skjuvbuckling.χ_w(1.5, 1.2, True), 0.62273)
kolla("χ_w λ̄_w = 1,5, ej styv", lambda: kb.skjuvbuckling.χ_w(1.5, 1.2, False), 0.55333)

## Flänsbidrag χ_f (5.4)

a = 2000, h_w = 1000, b_f = 300, t_f = 20, t_w = 8, S355, M_Ed = 1000 kNm.

M_f,Rd gäller bara flänsarna: b_f t_f f_y (h_w + t_f) = 2172,6 kNm. c = 548 mm, χ_f = 0,02157.

In [ ]:
kolla("χ_f klass 1", lambda: kb.skjuvbuckling.χ_f(1000e6, 2000, 1000, 300, 20, 8, 355, 355, 1)[0], 0.021573, känt_fel=True)
kolla("M_f,Rd klass 1", lambda: kb.skjuvbuckling.χ_f(1000e6, 2000, 1000, 300, 20, 8, 355, 355, 1)[1], 2172.6e6, känt_fel=True)
kolla_fel("χ_f klass 4 ger tydligt fel", lambda: kb.skjuvbuckling.χ_f(1000e6, 2000, 1000, 300, 20, 8, 355, 355, 4), känt_fel=True)

## χ_V och V_c,Rd

In [ ]:
kolla("χ_V", lambda: kb.skjuvbuckling.χ_V(0.62, 0.02, 1.2), 0.64)
kolla("χ_V ≤ η", lambda: kb.skjuvbuckling.χ_V(1.2, 0.1, 1.2), 1.2)
kolla("V_c,Rd", lambda: kb.skjuvbuckling.V_c_Rd(355, 1.0, 0.64, 1000, 8), 1049392)

## Sammanfattning

Cellen stoppar med `AssertionError` om någon kontroll som inte är märkt som känt fel misslyckas.

In [ ]:
sammanfattning()